[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA2/blob/main/en/lab6/lab6_part1.ipynb)
# Lab 6: Convolutional neural networks — Part 1 — FF vs CNN


### Prerequisites. Install packages

For the first part of this Lab 6 we will need, in addition to torch, the torchvision module to load the data. In addition, as usual, we will set the random seed to ensure the reproducibility of the experiments.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision
from torchvision import transforms
import os
import numpy as np
import random
from matplotlib import pyplot as plt

# We set the seed for reproducibility
seed = 1234567
os.environ['PYTHONHASHSEED'] = str(seed)
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)


### Loading the dataset

This time we will work with the *mnist* image dataset, which represents handwritten digits. We must tell our dataloader that it makes batches of 128 elements.

In [ ]:
# Load MNIST using torchvision
transform = transforms.Compose([
    transforms.ToTensor(), # convert to tensor and scale to [0,1]
])

train_val = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# TODO - Split train_val into train and val (80/20) (use torch data.random_split)

# TODO - Create the DataLoaders
batch_size = 128
train_loader = ...
val_loader = ...
test_loader = ...

NUM_CLASSES = 10
IMAGE_SHAPE = (1, 28, 28)

# To check loading worked, take one example and display it
im_batch, label_batch = next(iter(train_loader))
example_image = im_batch[0]
plt.imshow(example_image.squeeze())
plt.xlabel(f'Label: {label_batch[0].item()}')
plt.show()


## Fitting the data with a feed-forward neural network

We are going to model the data with a feed-forward network that has layers of 40, 25, and 16 units (all with ReLU activation). We must take into account that the images are 3-dimensional tensors (their `shape` is (1,28,28)), while the input of our Dense layers must be a 1-dimensional tensor. To adapt the input to what we need, we are going to "flatten" the image tensors, which will go from `shape` (28,28,1) to `shape` (784). To do this we will flatten the batch from (128,1,28,28) to (128, 784) in the `forward` function.

Finally, the output of our model must have as many components as distinct classes the set has. Since we want the output to approximate the probability of the different classes, the usual thing would be to use a *softmax* activation function, but in this case, for training efficiency reasons, it is better to keep a linear output and later use the `nn.CrossEntropyLoss` loss function, which expects the outputs to come in that format.

In [ ]:
class MLP(nn.Module):
    # TODO - Complete the class following the instructions given in the previous cell.

### Training the model

We are going to set the loss function, the optimizer (Adam with default LR) and the metric that will serve to evaluate the performance of the trained model (categorical precision).

Since we try to predict one class among several, our loss function must be the [cross entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html). We will tell it that the output of our network comes as *logits* and the labels with the numeric value of the class (not in one-hot encoding).

In [ ]:
# Evaluation on the test set
def evaluate(model, dataloader, loss_fn):
    # TODO - Complete the function so it returns loss and accuracy


def train_model(model, train_loader, val_loader, loss_fn, optimizer, num_epochs=16):
    # TODO - Complete the function so it trains and returns loss and accuracy histories

# TODO - Define the model, loss, and optimizer (Adam with lr=0.001) and train for 16 epochs
ff_model = ...
loss_fn = ...
optimizer = ...
train_losses, val_losses = ...

### Verifying the performance

We will use the test set to check the generalization ability of our model. You can also visualize the training curve to identify potential problems with the training.

In [ ]:
# TODO - Evaluate on the test set

If everything went correctly you should have obtained a precision value on the test set comparable to those obtained with the training and validation sets, which indicates that the model generalizes well to other data of the original set but... do we have a good model?

Let's check the robustness of the model by making small shifts of the original images. We will use a small model that applies a random translation of up to 10% of the image size to each of the test images. We will help ourselves with the `transforms` module included in `torchvision` and in particular with [`transforms.RandomAffine`](https://docs.pytorch.org/vision/main/generated/torchvision.transforms.RandomAffine.html) to make translations of up to 10% of the image size in any direction.

In [ ]:
# Data augmentation: apply a small translation using torchvision.transforms
transform_translated = transforms.Compose([
    # TODO - Use RandomAffine for translations of up to 10% in any direction
    transforms.ToTensor(),
])

# Create a transformed test dataset
test_translated = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform_translated)
test_translated_loader = ...

Let us now check the precision on this new set of images that have been slightly shifted.

In [ ]:
# TODO - Evaluate on the shifted test set

If everything went well, you should have verified that these small translations are enough to substantially lower the precision of the model. Feed-forward networks are not robust against this kind of perturbation.

## Comparison with a convolutional network

Now declare a convolutional model with the following architecture:
 1. [2D Convolution](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) of 8 filters and kernel size 3, with ReLU activation
 1. [2D Pooling](https://docs.pytorch.org/docs/2.8/generated/torch.nn.MaxPool2d.html) taking the maximum of each group of 2x2
  1. [2D Convolution](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) of 8 filters and kernel size 3, with ReLU activation
 1. [2D Pooling](https://docs.pytorch.org/docs/2.8/generated/torch.nn.MaxPool2d.html) taking the maximum of each group of 2x2
 1. Dense layer (requires prior flattening) of 32 units and ReLU activation
 1. Output layer

Define the new model, train it and do the subsequent verifications to observe the difference.

In [ ]:
class ConvModel(nn.Module):
    # TODO - Complete the class following the instructions

# TODO - Define the model, loss, and optimizer (Adam with lr=0.001) and train for 16 epochs
conv_model = ...
loss_fn = ...
optimizer = ...
train_losses, val_losses = ...

# TODO - Run the same evaluations as you did for ff_model

### Reflections on the comparison
 - What have you observed in the performance?
 - How many parameters does the convolutional network have compared to the *feed-forward*?
 - How has the execution time changed?
 - Is this network more robust to shifts?